In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# Ensure UTC+8 timezone object is available as required (not converting timestamps per instructions)
tz = pytz.timezone('Asia/Shanghai')

# Reuse variable names for efficiency across steps
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')

# Ensure 'value' is numeric (coerce errors to NaN)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# Function to produce grouped summary for a dataframe given KPI column name
def grouped_summary(df, kpi_col):
    group_on = ['cmdb_id', kpi_col]
    grouped = df.groupby(group_on).agg(
        count=('value', 'count'),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max'),
        p95=('value', lambda s: s.quantile(0.95) if s.notna().any() else np.nan),
        p05=('value', lambda s: s.quantile(0.05) if s.notna().any() else np.nan),
        mean=('value', 'mean')
    ).reset_index()
    grouped_sorted = grouped.sort_values('count', ascending=False)
    return grouped, grouped_sorted

# Metric file summary
metric_total_rows = metric_df.shape[0]
metric_full_grouped, metric_sorted = grouped_summary(metric_df, 'kpi_name')
metric_summary_top20 = metric_sorted.head(20)
metric_components = pd.Series(sorted(metric_df['cmdb_id'].dropna().unique()), name='components')
metric_kpis = pd.Series(sorted(metric_df['kpi_name'].dropna().unique()), name='kpi_names')

# Trace file summary
trace_total_rows = trace_df.shape[0]
trace_full_grouped, trace_sorted = grouped_summary(trace_df, 'trace_name')
trace_summary_top20 = trace_sorted.head(20)
trace_components = pd.Series(sorted(trace_df['cmdb_id'].dropna().unique()), name='components')
trace_kpis = pd.Series(sorted(trace_df['trace_name'].dropna().unique()), name='trace_names')

# Log file summary
log_total_rows = log_df.shape[0]
log_full_grouped, log_sorted = grouped_summary(log_df, 'log_name')
log_summary_top20 = log_sorted.head(20)
log_components = pd.Series(sorted(log_df['cmdb_id'].dropna().unique()), name='components')
log_kpis = pd.Series(sorted(log_df['log_name'].dropna().unique()), name='log_names')

# Compact outputs (variables shown; avoids printing raw full DataFrames)
metric_total_rows, metric_summary_top20, metric_components, metric_kpis, \
trace_total_rows, trace_summary_top20, trace_components, trace_kpis, \
log_total_rows, log_summary_top20, log_components, log_kpis

```
Out[1]:
```
```python
# Compose a concise plain-English summary using existing in-memory results
metric_rows = metric_total_rows
metric_comp_count = metric_components.shape[0]
metric_kpi_count = metric_kpis.shape[0]
metric_top = metric_summary_top20.iloc[:3] if metric_summary_top20.shape[0] >= 3 else metric_summary_top20

trace_rows = trace_total_rows
trace_comp_count = trace_components.shape[0]
trace_kpi_count = trace_kpis.shape[0]
trace_top = trace_summary_top20.iloc[:3] if trace_summary_top20.shape[0] >= 3 else trace_summary_top20

log_rows = log_total_rows
log_comp_count = log_components.shape[0]
log_kpi_count = log_kpis.shape[0]
log_top = log_summary_top20.iloc[:3] if log_summary_top20.shape[0] >= 3 else log_summary_top20

# Helper to format top group examples compactly
def fmt_top(df, kpi_col):
    lines = []
    for _, r in df.iterrows():
        lines.append(f"{r['cmdb_id']} | {r[kpi_col]} | count={int(r['count'])} | min_ts={int(r['min_timestamp'])} | max_ts={int(r['max_timestamp'])} | p95={r['p95']:.6g} | p05={r['p05']:.6g} | mean={r['mean']:.6g}")
    return lines

metric_examples = fmt_top(metric_top, 'kpi_name')
trace_examples = fmt_top(trace_top, 'trace_name')
log_examples = fmt_top(log_top, 'log_name')

summary = (
    "Summary of telemetry files (UTC+8 timestamps kept as integers):\n\n"
    f"1) metric.csv\n"
    f"- Total rows: {metric_rows}\n"
    f"- Unique components: {metric_comp_count}\n"
    f"- Unique KPI names: {metric_kpi_count}\n"
    f"- Top groups (by point count) — showing up to 3 examples:\n  - " + "\n  - ".join(metric_examples) + "\n\n"
    f"2) trace.csv\n"
    f"- Total rows: {trace_rows}\n"
    f"- Unique components: {trace_comp_count}\n"
    f"- Unique trace names: {trace_kpi_count}\n"
    f"- Top groups (by point count) — showing up to 3 examples:\n  - " + "\n  - ".join(trace_examples) + "\n\n"
    f"3) log.csv\n"
    f"- Total rows: {log_rows}\n"
    f"- Unique components: {log_comp_count}\n"
    f"- Unique log names: {log_kpi_count}\n"
    f"- Top groups (by point count) — showing up to 3 examples:\n  - " + "\n  - ".join(log_examples) + "\n\n"
    "Notes: For each group the aggregates shown are: count, min_timestamp, max_timestamp, global P95, global P05, and mean of 'value'.\n"
    "Full grouped tables and complete lists of components/KPIs are kept in memory; this summary is compact (top examples only)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(147308,                  cmdb_id                                           kpi_name  count  min_timestamp  max_timestamp           p95           p05          mean
4925  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     30     1647815400     1647817140   1845.725000    718.887500   1230.483333
0              adservice                                     app.grpc.count     30     1647815400     1647817140     84.000000     50.550000     73.133333
1              adservice                                       app.grpc.mrt     30     1647815400     1647817140      0.002768      0.002362      0.002538
2              adservice                                        app.grpc.rr     30     1647815400     1647817140    100.000000    100.000000    100.000000
3              adservice                                        app.grpc.sr     30     1647815400     1647817140     93.447183     77.631579     83.971152
4              adservice                                     app.http.count     30     1647815400     1647817140     24.000000     24.000000     24.000000
5              adservice                                       app.http.mrt     30     1647815400     1647817140      0.009163      0.008736      0.008961
6              adservice                                        app.http.rr     30     1647815400     1647817140    100.000000    100.000000    100.000000
7              adservice                                        app.http.sr     30     1647815400     1647817140    100.000000    100.000000    100.000000
8              adservice    runtime.java_lang_ClassLoading_LoadedClassCount     30     1647815400     1647817140   4990.000000   4990.000000   4990.000000
9              adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...     30     1647815400     1647817140   4990.000000   4990.000000   4990.000000
10             adservice  runtime.java_lang_ClassLoading_UnloadedClassCount     30     1647815400     1647817140      0.000000      0.000000      0.000000
11             adservice             runtime.java_lang_ClassLoading_Verbose     30     1647815400     1647817140      0.000000      0.000000      0.000000
12             adservice  runtime.java_lang_Compilation_CompilationTimeM...     30     1647815400     1647817140      1.000000      1.000000      1.000000
13             adservice  runtime.java_lang_Compilation_TotalCompilation...     30     1647815400     1647817140  43732.000000  43732.000000  43732.000000
14             adservice  runtime.java_lang_GarbageCollector_CollectionC...     30     1647815400     1647817140  43398.379167  43177.325000  43287.997222
15             adservice  runtime.java_lang_GarbageCollector_CollectionC...     30     1647815400     1647817140      3.000000      3.000000      3.000000
16             adservice  runtime.java_lang_GarbageCollector_CollectionT...     30     1647815400     1647817140  68142.225000  67800.154167  67971.516667
17             adservice  runtime.java_lang_GarbageCollector_CollectionT...     30     1647815400     1647817140    177.000000    177.000000    177.000000
18             adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...     30     1647815400     1647817140      1.000000      1.000000      1.000000, 0              adservice
1            adservice-0
2            adservice-1
3            adservice-2
4             adservice2
             ...        
54       shippingservice
55     shippingservice-0
56     shippingservice-1
57     shippingservice-2
58    shippingservice2-0
Name: components, Length: 59, dtype: object, 0               app.grpc.count
1                 app.grpc.mrt
2                  app.grpc.rr
3                  app.grpc.sr
4               app.http.count
                 ...          
1215         system.swap.total
1216          system.swap.used
1217      system.swap.used_pct
1218    system.tcp.retrans_pct
1219    system.udp.connect.num
Name: kpi_names, Length: 1220, dtype: object, 25060,                        cmdb_id                                      trace_name  count  min_timestamp  max_timestamp         p95        p05        mean
39                adservice2-0                trace.from_frontend2-0.row_count     30     1647815400     1647817140   51.000000  24.350000   37.433333
1239        shippingservice2-0                trace.from_frontend2-0.row_count     30     1647815400     1647817140   21.000000   5.000000   12.533333
1238        shippingservice2-0               trace.from_frontend2-0.error_rate     30     1647815400     1647817140    0.000000   0.000000    0.000000
1237        shippingservice2-0             trace.from_frontend2-0.duration_p95     30     1647815400     1647817140    0.000193   0.000063    0.000093
664                 frontend-1                        trace.self.duration_mean     30     1647815400     1647817140    0.440286   0.003631    0.157508
665                 frontend-1                         trace.self.duration_p95     30     1647815400     1647817140    0.007257   0.006208    0.506444
666                 frontend-1                           trace.self.error_rate     30     1647815400     1647817140    0.000000   0.000000    0.000000
667                 frontend-1                            trace.self.row_count     30     1647815400     1647817140  423.300000  38.500000  243.266667
708                 frontend-1        trace.to_currencyservice-1.duration_mean     30     1647815400     1647817140    0.000111   0.000088    0.000100
709                 frontend-1         trace.to_currencyservice-1.duration_p95     30     1647815400     1647817140    0.000130   0.000101    0.000126
710                 frontend-1           trace.to_currencyservice-1.error_rate     30     1647815400     1647817140    0.000000   0.000000    0.000000
711                 frontend-1            trace.to_currencyservice-1.row_count     30     1647815400     1647817140   47.550000   3.250000   24.300000
1136   recommendationservice-2  trace.to_productcatalogservice-2.duration_mean     30     1647815400     1647817140    0.000028   0.000022    0.000024
1137   recommendationservice-2   trace.to_productcatalogservice-2.duration_p95     30     1647815400     1647817140    0.000036   0.000024    0.000028
1138   recommendationservice-2     trace.to_productcatalogservice-2.error_rate     30     1647815400     1647817140    0.000000   0.000000    0.000000
1139   recommendationservice-2      trace.to_productcatalogservice-2.row_count     30     1647815400     1647817140    7.000000   2.000000    4.833333
1140  recommendationservice2-0            trace.from_frontend2-0.duration_mean     30     1647815400     1647817140    0.002735   0.002592    0.002657
1141  recommendationservice2-0             trace.from_frontend2-0.duration_p95     30     1647815400     1647817140    0.003418   0.002981    0.003163
1142  recommendationservice2-0               trace.from_frontend2-0.error_rate     30     1647815400     1647817140    0.000000   0.000000    0.000000
1143  recommendationservice2-0                trace.from_frontend2-0.row_count     30     1647815400     1647817140   58.550000  32.900000   44.966667, 0                  adservice-0
1                  adservice-1
2                  adservice-2
3                 adservice2-0
4                cartservice-0
                ...           
35    recommendationservice2-0
36           shippingservice-0
37           shippingservice-1
38           shippingservice-2
39          shippingservice2-0
Name: components, Length: 40, dtype: object, 0      trace.from_checkoutservice-0.duration_mean
1       trace.from_checkoutservice-0.duration_p95
2         trace.from_checkoutservice-0.error_rate
3          trace.from_checkoutservice-0.row_count
4      trace.from_checkoutservice-1.duration_mean
                          ...                    
195          trace.to_shippingservice-2.row_count
196     trace.to_shippingservice2-0.duration_mean
197      trace.to_shippingservice2-0.duration_p95
198        trace.to_shippingservice2-0.error_rate
199         trace.to_shippingservice2-0.row_count
Name: trace_names, Length: 200, dtype: object, 1614,                     cmdb_id         log_name  count  min_timestamp  max_timestamp     p95    p05        mean
0               adservice-0  log.error_count     30     1647815400     1647817140    0.00   0.00    0.000000
1               adservice-0    log.row_count     30     1647815400     1647817140  397.90  98.70  277.400000
2               adservice-1  log.error_count     30     1647815400     1647817140    0.00   0.00    0.000000
3               adservice-1    log.row_count     30     1647815400     1647817140   36.00   8.90   24.133333
4               adservice-2  log.error_count     30     1647815400     1647817140    0.00   0.00    0.000000
5               adservice-2    log.row_count     30     1647815400     1647817140   34.20   8.90   24.133333
6             cartservice-0  log.error_count     30     1647815400     1647817140    0.00   0.00    0.000000
7             cartservice-0    log.row_count     30     1647815400     1647817140  269.65  69.45  193.300000
8             cartservice-1  log.error_count     30     1647815400     1647817140    0.00   0.00    0.000000
9             cartservice-1    log.row_count     30     1647815400     1647817140  269.55  79.00  192.733333
10            cartservice-2  log.error_count     30     1647815400     1647817140    0.00   0.00    0.000000
11            cartservice-2    log.row_count     30     1647815400     1647817140  265.50  74.05  193.066667
20        currencyservice-1  log.error_count     30     1647815400     1647817140    0.00   0.00    0.000000
21        currencyservice-1    log.row_count     30     1647815400     1647817140  235.85  66.05  167.766667
23        currencyservice-2    log.row_count     30     1647815400     1647817140  239.55  64.85  168.133333
22        currencyservice-2  log.error_count     30     1647815400     1647817140    0.00   0.00    0.000000
19        currencyservice-0    log.row_count     30     1647815400     1647817140  238.75  66.65  168.400000
18        currencyservice-0  log.error_count     30     1647815400     1647817140    0.00   0.00    0.000000
50  recommendationservice-1  log.error_count     30     1647815400     1647817140    0.00   0.00    0.000000
51  recommendationservice-1    log.row_count     30     1647815400     1647817140   71.55  39.00   54.733333, 0                 adservice-0
1                 adservice-1
2                 adservice-2
3               cartservice-0
4               cartservice-1
               ...           
26    recommendationservice-2
27               redis-cart-0
28          shippingservice-0
29          shippingservice-1
30          shippingservice-2
Name: components, Length: 31, dtype: object, 0    log.error_count
1      log.row_count
Name: log_names, dtype: object)```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse existing metric_df and metric_full_grouped if present; otherwise load metric.csv
try:
    metric_df
    metric_full_grouped
except NameError:
    metric_df = pd.read_csv('metric.csv')
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
    # compute global p95/p05 per (cmdb_id, kpi_name)
    metric_full_grouped = metric_df.groupby(['cmdb_id', 'kpi_name']).agg(
        count=('value', 'count'),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max'),
        p95=('value', lambda s: s.quantile(0.95) if s.notna().any() else np.nan),
        p05=('value', lambda s: s.quantile(0.05) if s.notna().any() else np.nan),
        mean=('value', 'mean')
    ).reset_index()

# Ensure numeric
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# 1) Global P95/P05 already available in metric_full_grouped (ensure columns exist)
global_thresholds = metric_full_grouped[['cmdb_id', 'kpi_name', 'p95', 'p05']].copy()

# 2) Convert incident window (UTC+8) to epoch timestamps (seconds)
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(datetime(2022, 3, 21, 6, 30, 0))
end_local   = tz.localize(datetime(2022, 3, 21, 7, 0, 0))
start_ts = int(start_local.timestamp())
end_ts = int(end_local.timestamp())

# 3) Candidate root-cause components set (nodes, services, and pod names from task)
nodes = [f'node-{i}' for i in range(1,7)]
services = [
    'frontend','shippingservice','checkoutservice','currencyservice','adservice',
    'emailservice','cartservice','productcatalogservice','recommendationservice','paymentservice'
]
# pods list (from task)
pods = [
'frontend-0','frontend-1','frontend-2','frontend2-0',
'shippingservice-0','shippingservice-1','shippingservice-2','shippingservice2-0',
'checkoutservice-0','checkoutservice-1','checkoutservice-2','checkoutservice2-0',
'currencyservice-0','currencyservice-1','currencyservice-2','currencyservice2-0',
'adservice-0','adservice-1','adservice-2','adservice2-0',
'emailservice-0','emailservice-1','emailservice-2','emailservice2-0',
'cartservice-0','cartservice-1','cartservice-2','cartservice2-0',
'productcatalogservice-0','productcatalogservice-1','productcatalogservice-2','productcatalogservice2-0',
'recommendationservice-0','recommendationservice-1','recommendationservice-2','recommendationservice2-0',
'paymentservice-0','paymentservice-1','paymentservice-2','paymentservice2-0'
]

candidate_set = set(nodes) | set(services) | set(pods)

# Restrict to groups present in metric_full_grouped whose cmdb_id is in candidate_set
groups_of_interest = global_thresholds[global_thresholds['cmdb_id'].isin(candidate_set)].copy()
# If no such groups, results will be empty
# 2b) Filter metric rows to incident window (inclusive)
window_df = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# 4) Merge window rows with global thresholds to get p95/p05 per row, but only for groups_of_interest
window_merged = window_df.merge(groups_of_interest, on=['cmdb_id','kpi_name'], how='inner')

# Mark anomalies: value > p95 OR value < p05
# Safely compute comparisons even if p95/p05 are NaN
window_merged['is_high'] = window_merged['value'] > window_merged['p95']
window_merged['is_low'] = window_merged['value'] < window_merged['p05']
window_merged['is_anomaly'] = window_merged['is_high'] | window_merged['is_low']

# Compute deviation ratios for anomaly points
def compute_ratio(row):
    if not row['is_anomaly']:
        return np.nan
    # high anomaly
    if row['is_high']:
        p95 = row['p95']
        if pd.isna(p95) or p95 == 0:
            return np.nan
        return (row['value'] - p95) / p95
    # low anomaly
    if row['is_low']:
        p05 = row['p05']
        if pd.isna(p05) or p05 == 0:
            return np.nan
        return (p05 - row['value']) / p05
    return np.nan

window_merged['deviation_ratio'] = window_merged.apply(compute_ratio, axis=1)

# Aggregations per group across the window (including groups that may have zero anomalies)
group_cols = ['cmdb_id','kpi_name']
agg_all = window_merged.groupby(group_cols).agg(
    window_point_count = ('value','count'),
    max_value_in_window = ('value','max'),
    min_value_in_window = ('value','min'),
    anomaly_count = ('is_anomaly','sum'),
    earliest_anomaly_timestamp = ('timestamp', lambda s: int(s[window_merged.loc[s.index,'is_anomaly']].min()) if s[window_merged.loc[s.index,'is_anomaly']].any() else np.nan),
    global_p95 = ('p95','first'),
    global_p05 = ('p05','first'),
    max_deviation_ratio = ('deviation_ratio','max'),
    high_count = ('is_high','sum'),
    low_count = ('is_low','sum')
).reset_index()

# Determine anomaly_direction
def direction(row):
    if row['anomaly_count'] == 0:
        return 'none'
    high = row['high_count']
    low = row['low_count']
    if high>0 and low==0:
        return 'high'
    if low>0 and high==0:
        return 'low'
    return 'mixed'

agg_all['anomaly_direction'] = agg_all.apply(direction, axis=1)

# 5) Filter groups with >=1 anomaly
anomalous_groups = agg_all[agg_all['anomaly_count'] >= 1].copy()

# Compute total number of groups with >=1 anomaly
total_anomalous_groups = anomalous_groups.shape[0]

# Prepare final table, sort by anomaly_count desc then max_deviation_ratio desc
final_table = anomalous_groups.sort_values(
    by=['anomaly_count','max_deviation_ratio'],
    ascending=[False, False]
)[[
    'cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp',
    'max_value_in_window','min_value_in_window','global_p95','global_p05',
    'max_deviation_ratio','anomaly_direction'
]].reset_index(drop=True)

# Keep output compact: top 20 rows
anomaly_top20 = final_table.head(20)

# Display results compactly as variables
total_anomalous_groups, anomaly_top20, start_ts, end_ts, len(groups_of_interest)

```
Out[2]:
```
```python
# Summary in plain English based on the previous execution results (compact)
top = anomaly_top20.head(5)

summary = (
    f"Incident window (UTC+8): 2022-03-21 06:30 to 07:00 -> timestamps {start_ts} to {end_ts}.\n\n"
    f"Total candidate groups present for analysis: {len(groups_of_interest)}.\n"
    f"Total groups with >=1 anomaly in the window: {total_anomalous_groups}.\n\n"
    "Top anomaly groups (showing up to 5 examples from the top 20):\n"
)

lines = []
for _, r in top.iterrows():
    lines.append(
        f"- {r['cmdb_id']} | {r['kpi_name']} | anomalies={int(r['anomaly_count'])} | "
        f"earliest_ts={int(r['earliest_anomaly_timestamp'])} | max={r['max_value_in_window']:.6g} | "
        f"min={r['min_value_in_window']:.6g} | p95={r['global_p95']:.6g} | p05={r['global_p05']:.6g} | "
        f"max_dev_ratio={r['max_deviation_ratio']:.6g} | direction={r['anomaly_direction']}"
    )

summary += "\n".join(lines) + "\n\n"
summary += "Notes: Many top groups are container network receive metrics and some mesh/mesh-source metrics. Most top entries are classified as 'mixed' (both above-P95 and below-P05 events observed)."

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(1743,                      cmdb_id                                           kpi_name  anomaly_count  earliest_anomaly_timestamp  max_value_in_window  min_value_in_window  global_p95  global_p05  max_deviation_ratio anomaly_direction
0    recommendationservice-1  mesh.source.recommendationservice.productcatal...              4                1.647816e+09           628.008400             0.002100    1.142651    0.009975           548.606365             mixed
1                 frontend-1  mesh.source.frontend.recommendationservice.ist...              4                1.647816e+09            20.515250             0.003050    0.108841    0.010442           187.487821             mixed
2          paymentservice2-0  container.node-6.container_network_receive_MB....              4                1.647816e+09             0.574214             0.014949    0.026425    0.017574            20.730085             mixed
3             emailservice-2  container.node-5.container_network_receive_MB....              4                1.647816e+09             0.551733             0.018872    0.025924    0.019183            20.282562             mixed
4             emailservice-1  container.node-5.container_network_receive_MB....              4                1.647816e+09             0.552440             0.017729    0.027314    0.020052            19.225164             mixed
5            emailservice2-0  container.node-5.container_network_receive_MB....              4                1.647815e+09             0.556211             0.018517    0.027632    0.020473            19.129027             mixed
6          shippingservice-2  container.node-5.container_network_receive_MB....              4                1.647816e+09             0.549796             0.017759    0.029584    0.021205            17.584312             mixed
7          shippingservice-1  container.node-5.container_network_receive_MB....              4                1.647815e+09             0.551107             0.016621    0.029992    0.019807            17.375170             mixed
8                adservice-2  container.node-5.container_network_receive_MB....              4                1.647816e+09             0.575577             0.020842    0.034101    0.023769            15.878384             mixed
9    recommendationservice-1        container.node-5.container_cpu_user_seconds              4                1.647816e+09            10.415000             0.010000    0.634750    0.024500            15.408035             mixed
10   recommendationservice-1       container.node-5.container_cpu_usage_seconds              4                1.647816e+09            10.461374             0.042960    0.665096    0.046344            14.729130             mixed
11             cartservice-2  container.node-5.container_network_receive_MB....              4                1.647816e+09             0.561715             0.024928    0.041392    0.026070            12.570678             mixed
12          paymentservice-2  container.node-5.container_network_receive_MB....              4                1.647816e+09             0.286645             0.017561    0.024237    0.017875            10.826592             mixed
13                frontend-1  mesh.destination.unknown.frontend.istio_reques...              4                1.647816e+09            20.794750             0.096000    1.789525    0.193913            10.620262             mixed
14   recommendationservice-1  mesh.destination.frontend.recommendationservic...              4                1.647816e+09            13.526450             0.006100    1.213151    0.030225            10.149846             mixed
15         currencyservice-1  container.node-5.container_network_receive_MB....              4                1.647816e+09             0.568900             0.027333    0.063343    0.030991             7.981320             mixed
16   recommendationservice-2  container.node-5.container_network_receive_MB....              4                1.647816e+09             0.593890             0.033389    0.069000    0.041589             7.607111             mixed
17                frontend-0  container.node-5.container_network_receive_MB....              4                1.647815e+09             0.646143             0.022528    0.134721    0.023439             3.796143             mixed
18  recommendationservice2-0  container.node-6.container_network_receive_MB....              4                1.647815e+09             0.622196             0.077470    0.168066    0.084161             2.702095             mixed
19                frontend-2  container.node-5.container_network_receive_MB....              4                1.647816e+09             0.651534             0.028888    0.182107    0.056489             2.577763             mixed, 1647815400, 1647817200, 4569)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import re
import pytz
from datetime import datetime

# Try to reuse previously computed variables; if missing, recompute minimal required pieces
try:
    window_merged
    anomalous_groups
    final_table
except NameError:
    # Recompute necessary pieces from metric.csv
    metric_df = pd.read_csv('metric.csv')
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
    # global thresholds per group
    metric_full_grouped = metric_df.groupby(['cmdb_id','kpi_name']).agg(
        p95=('value', lambda s: s.quantile(0.95) if s.notna().any() else np.nan),
        p05=('value', lambda s: s.quantile(0.05) if s.notna().any() else np.nan)
    ).reset_index()
    # incident window UTC+8
    tz = pytz.timezone('Asia/Shanghai')
    start_local = tz.localize(datetime(2022,3,21,6,30,0))
    end_local   = tz.localize(datetime(2022,3,21,7,0,0))
    start_ts = int(start_local.timestamp())
    end_ts = int(end_local.timestamp())
    # candidate set from task
    nodes = [f'node-{i}' for i in range(1,7)]
    services = [
        'frontend','shippingservice','checkoutservice','currencyservice','adservice',
        'emailservice','cartservice','productcatalogservice','recommendationservice','paymentservice'
    ]
    pods = [
    'frontend-0','frontend-1','frontend-2','frontend2-0',
    'shippingservice-0','shippingservice-1','shippingservice-2','shippingservice2-0',
    'checkoutservice-0','checkoutservice-1','checkoutservice-2','checkoutservice2-0',
    'currencyservice-0','currencyservice-1','currencyservice-2','currencyservice2-0',
    'adservice-0','adservice-1','adservice-2','adservice2-0',
    'emailservice-0','emailservice-1','emailservice-2','emailservice2-0',
    'cartservice-0','cartservice-1','cartservice-2','cartservice2-0',
    'productcatalogservice-0','productcatalogservice-1','productcatalogservice-2','productcatalogservice2-0',
    'recommendationservice-0','recommendationservice-1','recommendationservice-2','recommendationservice2-0',
    'paymentservice-0','paymentservice-1','paymentservice-2','paymentservice2-0'
    ]
    candidate_set = set(nodes) | set(services) | set(pods)
    # Filter metric rows to window
    window_df = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()
    # Merge with thresholds and candidate groups
    groups_of_interest = metric_full_grouped[metric_full_grouped['cmdb_id'].isin(candidate_set)].copy()
    window_merged = window_df.merge(groups_of_interest, on=['cmdb_id','kpi_name'], how='inner')
    window_merged['is_high'] = window_merged['value'] > window_merged['p95']
    window_merged['is_low'] = window_merged['value'] < window_merged['p05']
    window_merged['is_anomaly'] = window_merged['is_high'] | window_merged['is_low']
    def compute_ratio(row):
        if not row['is_anomaly']:
            return np.nan
        if row['is_high']:
            p95 = row['p95']
            if pd.isna(p95) or p95 == 0:
                return np.nan
            return (row['value'] - p95) / p95
        if row['is_low']:
            p05 = row['p05']
            if pd.isna(p05) or p05 == 0:
                return np.nan
            return (p05 - row['value']) / p05
        return np.nan
    window_merged['deviation_ratio'] = window_merged.apply(compute_ratio, axis=1)
    anomalies_df = window_merged[window_merged['is_anomaly']].copy()
    # per-kpi anomalous groups
    per_kpi = anomalies_df.groupby(['cmdb_id','kpi_name']).agg(
        anomaly_count=('is_anomaly','sum'),
        earliest_ts=('timestamp','min'),
        latest_ts=('timestamp','max'),
        max_deviation_ratio=('deviation_ratio','max')
    ).reset_index()
    # per-component aggregated
    anomalous_groups = per_kpi.copy()
    final_table = anomalous_groups.sort_values(['anomaly_count','max_deviation_ratio'], ascending=[False,False]).reset_index(drop=True)

# At this point we should have anomalies_df, per_kpi (or anomalous_groups). If anomalies_df missing, create it:
if 'anomalies_df' not in globals():
    anomalies_df = window_merged[window_merged['is_anomaly']].copy()

# If no anomalies, prepare empty outputs
if anomalies_df.shape[0] == 0:
    comp_summary_top20 = pd.DataFrame(columns=[
        'cmdb_id','total_anomaly_count','number_of_distinct_kpis_with_anomalies',
        'earliest_anomaly_timestamp','latest_anomaly_timestamp','max_deviation_ratio_overall','top_3_kpis_by_anomaly_count'
    ])
    node_summary = pd.DataFrame(columns=[
        'node','total_anomaly_count','number_of_impacted_pods','earliest_anomaly_timestamp','max_deviation_ratio'
    ])
    overall_earliest_ts = np.nan
else:
    # Recompute per-kpi anomaly summary from anomalies_df to ensure timestamps and deviation ratios are accurate
    per_kpi = anomalies_df.groupby(['cmdb_id','kpi_name']).agg(
        anomaly_count=('is_anomaly','sum'),
        earliest_ts=('timestamp','min'),
        latest_ts=('timestamp','max'),
        max_deviation_ratio=('deviation_ratio','max')
    ).reset_index()

    # 1) Per-component summary
    # total_anomaly_count, number_of_distinct_kpis_with_anomalies, earliest, latest, max_deviation_ratio_overall
    comp_agg = per_kpi.groupby('cmdb_id').agg(
        total_anomaly_count = ('anomaly_count','sum'),
        number_of_distinct_kpis_with_anomalies = ('kpi_name','nunique'),
        earliest_anomaly_timestamp = ('earliest_ts','min'),
        latest_anomaly_timestamp = ('latest_ts','max'),
        max_deviation_ratio_overall = ('max_deviation_ratio','max')
    ).reset_index()

    # top_3_kpis_by_anomaly_count per cmdb_id
    per_kpi_sorted = per_kpi.sort_values(['cmdb_id','anomaly_count','max_deviation_ratio'], ascending=[True,False,False])
    topk = per_kpi_sorted.groupby('cmdb_id').head(3)
    topk_list = topk.groupby('cmdb_id').apply(lambda df: ",".join(df.sort_values(['anomaly_count','max_deviation_ratio'], ascending=[False,False])['kpi_name'].astype(str).tolist())).reset_index()
    topk_list.columns = ['cmdb_id','top_3_kpis_by_anomaly_count']

    comp_summary = comp_agg.merge(topk_list, on='cmdb_id', how='left')

    # Sort as requested and limit to top20
    comp_summary_top20 = comp_summary.sort_values(['total_anomaly_count','max_deviation_ratio_overall'], ascending=[False,False]).head(20).reset_index(drop=True)

    # 2) Per-node summary: extract node id from kpi_name pattern 'container.node-X'
    # Add node column to anomalies_df (extract first matching node)
    anomalies_df['node_num'] = anomalies_df['kpi_name'].str.extract(r'container\.node-(\d+)')
    anomalies_df['node'] = anomalies_df['node_num'].apply(lambda x: f'node-{int(x)}' if pd.notna(x) else np.nan)

    # Consider only nodes node-1..node-6
    anomalies_nodes = anomalies_df[anomalies_df['node'].isin([f'node-{i}' for i in range(1,7)])].copy()

    if anomalies_nodes.shape[0] == 0:
        node_summary = pd.DataFrame(columns=[
            'node','total_anomaly_count','number_of_impacted_pods','earliest_anomaly_timestamp','max_deviation_ratio'
        ])
    else:
        node_agg = anomalies_nodes.groupby('node').agg(
            total_anomaly_count = ('is_anomaly','sum'),
            number_of_impacted_pods = ('cmdb_id','nunique'),
            earliest_anomaly_timestamp = ('timestamp','min'),
            max_deviation_ratio = ('deviation_ratio','max')
        ).reset_index()

        node_summary = node_agg.sort_values('total_anomaly_count', ascending=False).reset_index(drop=True)

    # overall earliest anomaly timestamp across all anomalies
    overall_earliest_ts = int(anomalies_df['timestamp'].min())

# Final compact outputs
comp_summary_top20, node_summary.head(6), overall_earliest_ts

```
Out[3]:
```
Summary (plain English, compact):

Incident window earliest timestamp (overall anomalies): 1647815400 (2022-03-21 06:30:00 UTC+8).

1) Per-component top results (top 10 of the reported top-20)
- frontend2-0 — total_anomaly_count=275, distinct_kpis=77, earliest=1647815400, latest=1647817140, max_dev≈0.6595, top KPIs include container.node-6.container_network_receive_MB...
- frontend-2  — total_anomaly_count=272, distinct_kpis=82, max_dev≈2.5778
- frontend-1  — total_anomaly_count=258, distinct_kpis=78, max_dev≈187.488 (mesh-source KPI among top)
- checkoutservice2-0 — total_anomaly_count=172, distinct_kpis=65, max_dev≈1.4098
- productcatalogservice-2 — total_anomaly_count=168, distinct_kpis=46, max_dev≈1.4394
- productcatalogservice-0 — total_anomaly_count=165, distinct_kpis=48, max_dev≈1.0000
- productcatalogservice-1 — total_anomaly_count=154, distinct_kpis=46, max_dev≈1.0000
- checkoutservice-2 — total_anomaly_count=144, distinct_kpis=63, max_dev≈425.1905
- frontend-0 — total_anomaly_count=139, distinct_kpis=74, max_dev≈3.7961
- currencyservice2-0 — total_anomaly_count=136, distinct_kpis=36, max_dev≈1.0096

(Other components up to rank 20 include productcatalogservice2-0, checkoutservice-0, node-6, cartservice2-0, adservice, node-1, node-2, cartservice-2, cartservice-0, shippingservice2-0.)

Notes:
- Earliest and latest anomaly timestamps for most top components are within 1647815400–1647817140 (the incident window analyzed).
- Several high-ranked KPIs are container network receive metrics and mesh-related (mesh.source / mesh.destination) metrics; some groups show very large max deviation ratios (>>1), indicating large spikes relative to their global P95/P05.

2) Per-node summary (nodes that appeared)
- node-5 — total_anomaly_count=1636, number_of_impacted_pods=33, earliest_anomaly_timestamp=1647815400, max_deviation_ratio≈163.5161
- node-6 — total_anomaly_count=456, number_of_impacted_pods=7, earliest_anomaly_timestamp=1647815400, max_deviation_ratio≈20.7301

Interpretation highlights (concise):
- Node-5 shows by far the largest anomaly volume (1636 anomalies across many pods/KPIs) and the highest single-point deviation ratio — it is the most impacted node in this window.
- Node-6 is the second-most impacted node but at substantially lower anomaly count.
- Multiple service pods (frontend, productcatalogservice, checkoutservice, etc.) exhibit many anomalous KPI points; many anomalies are classified as large deviations in network/mesh-related KPIs.
- Overall earliest anomaly observed across all groups: 1647815400 (2022-03-21 06:30:00 UTC+8).

If you want, I can export the full top-20 per-component table and the per-node table in a compact textual CSV-style format.

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id  total_anomaly_count  number_of_distinct_kpis_with_anomalies  earliest_anomaly_timestamp  latest_anomaly_timestamp  max_deviation_ratio_overall                        top_3_kpis_by_anomaly_count
0                frontend2-0                  275                                      77                  1647815400                1647817140                     0.659479  container.node-6.container_network_receive_MB....
1                 frontend-2                  272                                      82                  1647815400                1647817140                     2.577763  container.node-5.container_network_receive_MB....
2                 frontend-1                  258                                      78                  1647815400                1647817140                   187.487821  mesh.source.frontend.recommendationservice.ist...
3         checkoutservice2-0                  172                                      65                  1647815400                1647817140                     1.409759  mesh.source.checkoutservice2.currencyservice2....
4    productcatalogservice-2                  168                                      46                  1647815400                1647817140                     1.439404  container.node-5.container_network_receive_MB....
5    productcatalogservice-0                  165                                      48                  1647815400                1647817140                     1.000000  mesh.destination.checkoutservice.productcatalo...
6    productcatalogservice-1                  154                                      46                  1647815400                1647817140                     1.000000  mesh.destination.checkoutservice.productcatalo...
7          checkoutservice-2                  144                                      63                  1647815400                1647817140                   425.190476  container.node-5.container_network_receive_MB....
8                 frontend-0                  139                                      74                  1647815400                1647817140                     3.796143  container.node-5.container_network_receive_MB....
9         currencyservice2-0                  136                                      36                  1647815400                1647817140                     1.009625  container.node-6.container_network_receive_MB....
10  productcatalogservice2-0                  131                                      36                  1647815400                1647817140                     1.789438  mesh.source.productcatalogservice2.jaeger-coll...
11         checkoutservice-0                  129                                      65                  1647815400                1647817140                   187.459502  container.node-5.container_memory_failures.con...
12                    node-6                  128                                      36                  1647815400                1647817140                     1.000000      system.io.w_await,system.io.w_s,system.load.1
13            cartservice2-0                  119                                      33                  1647815400                1647817140                     0.757188  container.node-6.container_memory_failures.con...
14                 adservice                  119                                      34                  1647815400                1647817140                     0.626309  runtime.java_lang_OperatingSystem_SystemLoadAv...
15                    node-1                  118                                      36                  1647815400                1647817140                     2.213778  system.net.tcp.retrans_segs,system.net.bytes_s...
16                    node-2                  118                                      37                  1647815400                1647817020                     1.614379   system.io.await,system.io.w_await,system.io.util
17             cartservice-2                  112                                      32                  1647815400                1647817140                    12.570678  container.node-5.container_network_receive_MB....
18             cartservice-0                  111                                      30                  1647815400                1647817140                     1.000000  mesh.destination.checkoutservice.cartservice.i...
19        shippingservice2-0                  110                                      35                  1647815400                1647817140                     1.000000  container.node-5.container_memory_failures.con...,      node  total_anomaly_count  number_of_impacted_pods  earliest_anomaly_timestamp  max_deviation_ratio
0  node-5                 1636                       33                  1647815400           163.516129
1  node-6                  456                        7                  1647815400            20.730085, 1647815400)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse existing metric_df/anomalies_df if present; otherwise load and compute minimal anomalies for metric.csv
try:
    anomalies_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
    # global thresholds per (cmdb_id,kpi_name)
    metric_thresholds = metric_df.groupby(['cmdb_id','kpi_name']).agg(
        p95=('value', lambda s: s.quantile(0.95) if s.notna().any() else np.nan),
        p05=('value', lambda s: s.quantile(0.05) if s.notna().any() else np.nan)
    ).reset_index()
    # incident window UTC+8
    tz = pytz.timezone('Asia/Shanghai')
    start_local = tz.localize(datetime(2022,3,21,6,30,0))
    end_local   = tz.localize(datetime(2022,3,21,7,0,0))
    start_ts = int(start_local.timestamp())
    end_ts = int(end_local.timestamp())
    # filter metric rows to window and merge thresholds
    metric_window = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()
    window_merged = metric_window.merge(metric_thresholds, on=['cmdb_id','kpi_name'], how='inner')
    window_merged['is_high'] = window_merged['value'] > window_merged['p95']
    window_merged['is_low'] = window_merged['value'] < window_merged['p05']
    window_merged['is_anomaly'] = window_merged['is_high'] | window_merged['is_low']
    # compute deviation ratio for completeness (not required for this step)
    def compute_ratio(row):
        if not row['is_anomaly']:
            return np.nan
        if row['is_high']:
            p95 = row['p95']
            return (row['value'] - p95) / p95 if pd.notna(p95) and p95!=0 else np.nan
        if row['is_low']:
            p05 = row['p05']
            return (p05 - row['value']) / p05 if pd.notna(p05) and p05!=0 else np.nan
        return np.nan
    window_merged['deviation_ratio'] = window_merged.apply(compute_ratio, axis=1)
    anomalies_df = window_merged[window_merged['is_anomaly']].copy()

# 1) Extract distinct cmdb_id values that had anomaly points where kpi_name contains 'container.node-5'
mask_node5 = anomalies_df['kpi_name'].str.contains('container.node-5', na=False)
node5_cmdb_ids = pd.Series(sorted(anomalies_df.loc[mask_node5, 'cmdb_id'].dropna().unique()), name='cmdb_id').head(200)

# 2) Trace analysis
# Load trace.csv (reuse if already in memory)
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Compute global P95/P05 per (cmdb_id, trace_name) across full series (do NOT filter first)
trace_thresholds = trace_df.groupby(['cmdb_id','trace_name']).agg(
    global_p95 = ('value', lambda s: s.quantile(0.95) if s.notna().any() else np.nan),
    global_p05 = ('value', lambda s: s.quantile(0.05) if s.notna().any() else np.nan)
).reset_index()

# Incident window timestamps (reuse start_ts/end_ts if present)
try:
    start_ts, end_ts
except NameError:
    tz = pytz.timezone('Asia/Shanghai')
    start_local = tz.localize(datetime(2022,3,21,6,30,0))
    end_local   = tz.localize(datetime(2022,3,21,7,0,0))
    start_ts = int(start_local.timestamp())
    end_ts = int(end_local.timestamp())

# Filter trace rows to window and to cmdb_id list obtained in step 1
trace_window = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)].copy()
# If node5_cmdb_ids is empty, ensure it's an empty list
node5_list = node5_cmdb_ids.tolist() if not node5_cmdb_ids.empty else []
trace_window_node5 = trace_window[trace_window['cmdb_id'].isin(node5_list)].copy()

# Merge with global thresholds
trace_merged = trace_window_node5.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')

# Determine anomalies per row: value > global_p95 OR value < global_p05 (safely handle NaNs)
trace_merged['is_high'] = trace_merged['value'] > trace_merged['global_p95']
trace_merged['is_low'] = trace_merged['value'] < trace_merged['global_p05']
trace_merged['is_anomaly'] = trace_merged['is_high'] | trace_merged['is_low']

# Group per (cmdb_id, trace_name) and compute requested aggregates
def earliest_anom_ts(series):
    if series.any():
        return int(series.index[series.argmax()]) if False else int(series.index[0])  # placeholder, replaced below

# We'll compute using groupby with lambdas that use the merged frame
grouped = trace_merged.groupby(['cmdb_id','trace_name'])

agg_df = grouped.agg(
    anomaly_count = ('is_anomaly','sum'),
    earliest_anomaly_timestamp = ('timestamp', lambda s: int(s[trace_merged.loc[s.index,'is_anomaly']].min()) if s[trace_merged.loc[s.index,'is_anomaly']].any() else np.nan),
    max_value_in_window = ('value','max'),
    min_value_in_window = ('value','min'),
    global_p95 = ('global_p95','first'),
    global_p05 = ('global_p05','first'),
    high_count = ('is_high','sum'),
    low_count = ('is_low','sum')
).reset_index()

# anomaly_direction
def get_direction(r):
    if r['anomaly_count'] == 0:
        return 'none'
    if r['high_count']>0 and r['low_count']==0:
        return 'high'
    if r['low_count']>0 and r['high_count']==0:
        return 'low'
    return 'mixed'

agg_df['anomaly_direction'] = agg_df.apply(get_direction, axis=1)

# Filter to groups with anomaly_count > 0
agg_anomalies = agg_df[agg_df['anomaly_count'] > 0].copy()

# Sort by anomaly_count desc then max_value_in_window desc, limit top 20
agg_anomalies_sorted = agg_anomalies.sort_values(by=['anomaly_count','max_value_in_window'], ascending=[False, False]).head(20).reset_index(drop=True)

# earliest anomaly timestamp among all returned trace anomalies
if agg_anomalies.shape[0] > 0:
    earliest_trace_anom_ts = int(agg_anomalies['earliest_anomaly_timestamp'].min())
else:
    earliest_trace_anom_ts = np.nan

# Compact outputs: (a) node5 cmdb list, (b) top20 trace anomaly groups, (c) earliest timestamp
node5_cmdb_ids, agg_anomalies_sorted, earliest_trace_anom_ts

```
Out[4]:
```
```python
# Compose a plain-English summary string using previously computed variables.
import pytz
from datetime import datetime

# Reuse variables from prior computations: node5_cmdb_ids, agg_anomalies_sorted, earliest_trace_anom_ts
# If any are missing, raise an informative error.
try:
    node5_list = node5_cmdb_ids.tolist()
    trace_table = agg_anomalies_sorted.copy()
    earliest_ts = earliest_trace_anom_ts
except NameError as e:
    raise RuntimeError("Required intermediate results not found in memory. Please re-run the analysis steps.") from e

# Prepare node5 list summary (show up to first 20 items, report total count)
total_node5 = len(node5_list)
node5_display = node5_list[:20]
node5_line = ", ".join(node5_display) + (f", ... (total {total_node5})" if total_node5 > 20 else "")

# Prepare top trace anomaly groups lines (limit to 20)
lines = []
for _, r in trace_table.head(20).iterrows():
    lines.append(
        f"{r['cmdb_id']} | {r['trace_name']} | anomalies={int(r['anomaly_count'])} | "
        f"earliest_ts={int(r['earliest_anomaly_timestamp'])} | max={r['max_value_in_window']:.6g} | "
        f"min={r['min_value_in_window']:.6g} | p95={r['global_p95']:.6g} | p05={r['global_p05']:.6g} | dir={r['anomaly_direction']}"
    )

# Convert earliest timestamp to human-readable UTC+8 if present
if pd.isna(earliest_ts):
    earliest_hr = "None"
else:
    tz = pytz.timezone('Asia/Shanghai')
    earliest_hr = datetime.fromtimestamp(int(earliest_ts), tz).strftime("%Y-%m-%d %H:%M:%S %Z")

summary = (
    "Trace-anomaly follow-up summary (based on node-5 related metric anomalies):\n\n"
    "1) Distinct cmdb_id tied to node-5 (from metric anomalies):\n"
    f"- Count: {total_node5}\n"
    f"- Examples (up to 20): {node5_line}\n\n"
    "2) Top trace (cmdb_id, trace_name) groups with anomalies in the incident window (top 20):\n"
    + "\n".join(f"- {l}" for l in lines) + "\n\n"
    f"3) Earliest trace anomaly timestamp among these groups: {int(earliest_ts) if not pd.isna(earliest_ts) else 'None'} "
    f"(UTC+8: {earliest_hr})\n\n"
    "Notes: All global P95/P05 thresholds were computed across full trace series before window filtering. "
    "Trace anomaly direction values are one of: high, low, mixed, or none."
)

# Display the summary string (compact, plain English)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(0                 adservice-0
1                 adservice-1
2                 adservice-2
3               cartservice-0
4               cartservice-1
               ...           
28    recommendationservice-2
29          shippingservice-0
30          shippingservice-1
31          shippingservice-2
32         shippingservice2-0
Name: cmdb_id, Length: 33, dtype: object,                     cmdb_id                                     trace_name  anomaly_count  earliest_anomaly_timestamp  max_value_in_window  min_value_in_window  global_p95  global_p05  high_count  low_count anomaly_direction
0                frontend-1                           trace.self.row_count              4                1.647815e+09           432.000000            11.000000  423.300000   38.500000           2          2             mixed
1   recommendationservice-1                        trace.self.duration_p95              4                1.647816e+09           199.081207             0.001918   72.599160    0.001933           2          2             mixed
2                frontend-2  trace.to_recommendationservice-1.duration_p95              4                1.647816e+09           154.794010             0.002073   72.972221    0.002089           2          2             mixed
3   recommendationservice-1             trace.from_frontend-2.duration_p95              4                1.647816e+09           154.794010             0.002073   72.972221    0.002089           2          2             mixed
4                frontend-1  trace.to_recommendationservice-1.duration_p95              4                1.647816e+09           153.418639             0.002013    0.002644    0.002066           2          2             mixed
5   recommendationservice-1             trace.from_frontend-1.duration_p95              4                1.647816e+09           153.418639             0.002013    0.002644    0.002066           2          2             mixed
6                frontend-2     trace.to_productcatalogservice-0.row_count              4                1.647816e+09            75.000000             5.000000   69.100000   12.450000           2          2             mixed
7   productcatalogservice-0                trace.from_frontend-2.row_count              4                1.647816e+09            75.000000             5.000000   69.100000   12.450000           2          2             mixed
8                frontend-2     trace.to_productcatalogservice-1.row_count              4                1.647816e+09            74.000000             4.000000   70.100000   12.350000           2          2             mixed
9                frontend-2     trace.to_productcatalogservice-2.row_count              4                1.647816e+09            74.000000             4.000000   70.100000   12.350000           2          2             mixed
10  productcatalogservice-1                trace.from_frontend-2.row_count              4                1.647816e+09            74.000000             4.000000   70.100000   12.350000           2          2             mixed
11  productcatalogservice-2                trace.from_frontend-2.row_count              4                1.647816e+09            74.000000             4.000000   70.100000   12.350000           2          2             mixed
12               frontend-1     trace.to_productcatalogservice-2.row_count              4                1.647816e+09            71.000000             2.000000   63.100000    3.900000           2          2             mixed
13  productcatalogservice-2                trace.from_frontend-1.row_count              4                1.647816e+09            71.000000             2.000000   63.100000    3.900000           2          2             mixed
14               frontend-1     trace.to_productcatalogservice-0.row_count              4                1.647816e+09            70.000000             2.000000   63.100000    3.350000           2          2             mixed
15               frontend-1     trace.to_productcatalogservice-1.row_count              4                1.647816e+09            70.000000             2.000000   63.550000    3.800000           2          2             mixed
16  productcatalogservice-0                trace.from_frontend-1.row_count              4                1.647816e+09            70.000000             2.000000   63.100000    3.350000           2          2             mixed
17  productcatalogservice-1                trace.from_frontend-1.row_count              4                1.647816e+09            70.000000             2.000000   63.550000    3.800000           2          2             mixed
18               frontend-1                   trace.from_root.duration_p95              4                1.647816e+09            60.000320             0.047498   37.290447    0.049686           2          2             mixed
19               frontend-2                   trace.from_root.duration_p95              4                1.647816e+09            60.000314             0.048007   44.589973    0.049136           2          2             mixed, 1647815400)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse metric_df if present, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Incident window in UTC+8
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(datetime(2022,3,21,6,30,0))
end_local   = tz.localize(datetime(2022,3,21,7,0,0))
start_ts = int(start_local.timestamp())
end_ts = int(end_local.timestamp())

# 1) Identify rows where cmdb_id == 'node-5' OR kpi_name contains 'container.node-5'
mask_node5_rows = (metric_df['cmdb_id'] == 'node-5') | (metric_df['kpi_name'].str.contains('container.node-5', na=False))
node5_subset_full = metric_df[mask_node5_rows].copy()

# If no rows, prepare empty outputs
if node5_subset_full.shape[0] == 0:
    result_df = pd.DataFrame(columns=[
        'kpi_name','group_count','min_timestamp','max_timestamp','global_p95','global_p05','mean',
        'anomaly_count','earliest_anomaly_timestamp','max_value_in_window','min_value_in_window','anomaly_direction'
    ])
    network_issue_flag = False
    matching_kpis = pd.Series(dtype=str)
else:
    # 2) For each distinct kpi_name in subset compute global stats using the full available series for that kpi_name
    # Note: thresholds computed across all rows in node5_subset_full for each kpi_name (as required)
    group_stats = node5_subset_full.groupby('kpi_name', dropna=False).agg(
        group_count = ('value','count'),
        min_timestamp = ('timestamp','min'),
        max_timestamp = ('timestamp','max'),
        global_p95 = ('value', lambda s: s.quantile(0.95) if s.notna().any() else np.nan),
        global_p05 = ('value', lambda s: s.quantile(0.05) if s.notna().any() else np.nan),
        mean = ('value','mean')
    ).reset_index()

    # 2b) Restrict to incident window and compute anomalies per kpi_name
    window_mask = (metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)
    node5_window = metric_df[window_mask & mask_node5_rows].copy()

    # Merge window rows with global thresholds by kpi_name (thresholds computed above)
    node5_window = node5_window.merge(
        group_stats[['kpi_name','global_p95','global_p05']],
        on='kpi_name', how='left'
    )

    # Determine anomalies in window: value > global_p95 OR value < global_p05
    node5_window['is_high'] = node5_window['value'] > node5_window['global_p95']
    node5_window['is_low'] = node5_window['value'] < node5_window['global_p05']
    node5_window['is_anomaly'] = node5_window['is_high'] | node5_window['is_low']

    # Aggregate per kpi_name in window
    agg_window = node5_window.groupby('kpi_name', dropna=False).agg(
        anomaly_count = ('is_anomaly','sum'),
        earliest_anomaly_timestamp = ('timestamp', lambda s: int(s[node5_window.loc[s.index,'is_anomaly']].min()) if s[node5_window.loc[s.index,'is_anomaly']].any() else np.nan),
        max_value_in_window = ('value','max'),
        min_value_in_window = ('value','min'),
        high_count = ('is_high','sum'),
        low_count = ('is_low','sum')
    ).reset_index()

    # Determine anomaly_direction
    def direction(row):
        if row['anomaly_count'] == 0:
            return 'none'
        high = row['high_count']
        low = row['low_count']
        if high>0 and low==0:
            return 'high'
        if low>0 and high==0:
            return 'low'
        return 'mixed'
    agg_window['anomaly_direction'] = agg_window.apply(direction, axis=1)

    # Merge full-series group_stats with window aggregates
    merged = group_stats.merge(
        agg_window[['kpi_name','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','min_value_in_window','anomaly_direction']],
        on='kpi_name', how='left'
    )

    # Fill NaNs for KPIs that had no window rows
    merged['anomaly_count'] = merged['anomaly_count'].fillna(0).astype(int)
    merged['earliest_anomaly_timestamp'] = merged['earliest_anomaly_timestamp'].where(pd.notna(merged['earliest_anomaly_timestamp']), np.nan)
    merged['max_value_in_window'] = merged['max_value_in_window']
    merged['min_value_in_window'] = merged['min_value_in_window']
    merged['anomaly_direction'] = merged['anomaly_direction'].fillna('none')

    # Select and order columns as requested
    result_df = merged[[
        'kpi_name','group_count','min_timestamp','max_timestamp','global_p95','global_p05','mean',
        'anomaly_count','earliest_anomaly_timestamp','max_value_in_window','min_value_in_window','anomaly_direction'
    ]].copy()

    # Replace NaN max/min in window with NaN (explicit) and ensure numeric types
    result_df['max_value_in_window'] = pd.to_numeric(result_df['max_value_in_window'], errors='coerce')
    result_df['min_value_in_window'] = pd.to_numeric(result_df['min_value_in_window'], errors='coerce')

    # 3) Return top 20 kpi_name rows sorted by anomaly_count desc then by max_value_in_window desc
    result_df = result_df.sort_values(by=['anomaly_count','max_value_in_window'], ascending=[False, False]).head(20).reset_index(drop=True)

    # Additionally compute scalar summary: whether any kpi_name indicates network/tcp/retrans/packet/drop/err
    keywords = ['retrans','retrans_pct','packet','tcp','net','network','drop','err']
    pattern = '|'.join(keywords)
    matching_mask = result_df['kpi_name'].str.contains(pattern, case=False, na=False)
    # Also check all kpi_names in the full node5 subset for a broader flag
    all_kpi_names_node5 = node5_subset_full['kpi_name'].dropna().unique()
    broader_match_mask = pd.Series(all_kpi_names_node5).str.contains(pattern, case=False, na=False)
    matching_kpis = pd.Series(sorted(pd.Series(all_kpi_names_node5)[broader_match_mask].unique()), name='matching_kpis').head(200)
    network_issue_flag = bool(broader_match_mask.any())

# Final compact outputs: result_df (top up to 20 rows), network_issue_flag, matching_kpis (up to 200)
result_df, network_issue_flag, matching_kpis

```
Out[5]:
```
Summary (compact, plain English):

Incident window: 2022-03-21 06:30–07:00 (UTC+8).

Top findings for node-5 and its pods (top 20 KPI names shown; columns: kpi_name | group_count | global_p95 | global_p05 | anomaly_count | anomaly_direction):
- container.node-5.container_network_transmit_MB... — group_count=1050, p95≈2.70845, p05≈0.119241, anomaly_count=106, direction=mixed
- container.node-5.container_network_receive_MB... — group_count=1050, p95≈0.313165, p05≈0.019980, anomaly_count=106, direction=mixed
- container.node-5.container_network_receive_packets... — group_count=1050, p95≈1465.5, p05≈229.0, anomaly_count=104, direction=mixed
- container.node-5.container_network_transmit_packets... — group_count=1050, p95≈1383.35, p05≈155.0, anomaly_count=104, direction=mixed
- container.node-5.container_cpu_usage_seconds — group_count=990, p95≈0.236196, p05≈0.0005669, anomaly_count=100, direction=mixed
- container.node-5.container_memory_usage_MB — group_count=990, p95≈176.2656, p05≈23.329, anomaly_count=91, direction=mixed
- container.node-5.container_memory_working_set_MB — group_count=990, p95≈163.1055, anomaly_count=91, direction=mixed
- container.node-5.container_cpu_cfs_periods — group_count=990, p95≈140.775, anomaly_count=81, direction=mixed
- container.node-5.container_memory_rss — group_count=990, p95≈1.369e8, anomaly_count=80, direction=mixed
- container.node-5.container_fs_usage_MB./dev/vda1 — group_count=990, p95≈112.2764, anomaly_count=80, direction=mixed
- ... (rows 11–20 include socket/last_seen/start_time/memory_cache/memory_max_usage/memory_failures/cpu throttled/cpu_user_seconds/cpu_system_seconds — see full result if needed)

Notable node-level KPI anomalies:
- Some KPIs show many anomalies (100+ anomalies for network throughput and packet metrics; 90+ for memory/CPU metrics).
- Several KPIs are flagged with direction "high" for clear one-sided spikes: e.g., container_memory_failures.*, container_cpu_cfs_throttled_seconds, container_cpu_user_seconds, container_cpu_system_seconds. These indicate resource failures/throttling and elevated CPU activity.

Network/packet/tcp issues summary:
- A scan for keywords (retrans, retrans_pct, packet, tcp, net, network, drop, err) returned matches — network-related KPIs are present.
- matching_kpis count = 24 (examples: container.node-5.container_network_receive_MB..., container.node-5.container_network_receive_errors/packets, system.net.* metrics, system.tcp.retrans_pct).
- network_issue_flag = True (i.e., there are KPIs that indicate retransmissions/packet/network-related metrics).

Interpretation (concise):
- Node-5 and the pods on node-5 show substantial anomalies during the incident window, especially in network transmit/receive (throughput and packet counts) and also in CPU and memory metrics.
- Presence of high counts in container_network_* and packet metrics, plus system net/tcp-related KPIs, suggests network-level issues on node-5 alongside resource (CPU/memory/throttling/failures) symptoms.

If you want, I can produce the full top-20 KPI table in CSV-style text or filter it further by anomaly_direction (e.g., only one-sided high anomalies).

The original code execution output of IPython Kernel is also provided below for reference:

(                                             kpi_name  group_count  min_timestamp  max_timestamp    global_p95    global_p05          mean  anomaly_count  earliest_anomaly_timestamp  max_value_in_window  min_value_in_window anomaly_direction
0   container.node-5.container_network_transmit_MB...         1050     1647815400     1647817140  2.708450e+00  1.192413e-01  8.506241e-01            106                1.647815e+09         4.437957e+00         9.095955e-02             mixed
1   container.node-5.container_network_receive_MB....         1050     1647815400     1647817140  3.131651e-01  1.998012e-02  7.442431e-02            106                1.647815e+09         8.458552e-01         1.662064e-02             mixed
2   container.node-5.container_network_receive_pac...         1050     1647815400     1647817140  1.465500e+03  2.290000e+02  4.554343e+02            104                1.647815e+09         2.177500e+03         1.835000e+02             mixed
3   container.node-5.container_network_transmit_pa...         1050     1647815400     1647817140  1.383350e+03  1.550000e+02  3.784167e+02            104                1.647815e+09         2.106500e+03         1.055000e+02             mixed
4        container.node-5.container_cpu_usage_seconds          990     1647815400     1647817140  2.361960e-01  5.669092e-04  1.327332e-01            100                1.647815e+09         2.449639e+01         0.000000e+00             mixed
5          container.node-5.container_memory_usage_MB          990     1647815400     1647817140  1.762656e+02  2.332900e+01  7.570113e+01             91                1.647815e+09         2.032773e+02         2.156250e+01             mixed
6    container.node-5.container_memory_working_set_MB          990     1647815400     1647817140  1.631055e+02  2.316104e+01  6.877057e+01             91                1.647815e+09         1.840312e+02         2.143750e+01             mixed
7          container.node-5.container_cpu_cfs_periods          990     1647815400     1647817140  1.407750e+02  1.000000e+00  5.354343e+01             81                1.647815e+09         6.160000e+02         0.000000e+00             mixed
8               container.node-5.container_memory_rss          990     1647815400     1647817140  1.368965e+08  1.365565e+07  4.278520e+07             80                1.647815e+09         1.469972e+08         1.325875e+07             mixed
9    container.node-5.container_fs_usage_MB./dev/vda1          990     1647815400     1647817140  1.122764e+02  1.601562e-01  5.633416e+01             80                1.647815e+09         1.127012e+02         1.562500e-01             mixed
10                 container.node-5.container_sockets          990     1647815400     1647817140  1.000000e+01  2.500000e+00  4.733838e+00             72                1.647815e+09         1.200000e+01         1.000000e+00             mixed
11               container.node-5.container_last_seen          990     1647815400     1647817140  1.647817e+09  1.647816e+09  1.647816e+09             66                1.647815e+09         1.647817e+09         1.647815e+09             mixed
12      container.node-5.container_start_time_seconds          990     1647815400     1647817140  1.647778e+09  1.647523e+09  1.647546e+09             60                1.647815e+09         1.647778e+09         1.647523e+09             mixed
13            container.node-5.container_memory_cache          990     1647815400     1647817140  4.525261e+07  1.474560e+05  2.222137e+07             60                1.647815e+09         5.566874e+07         1.638400e+04             mixed
14     container.node-5.container_memory_max_usage_MB          990     1647815400     1647817140  9.000000e+02  2.817188e+01  3.576725e+02             60                1.647815e+09         1.025000e+03         2.651953e+01             mixed
15  container.node-5.container_memory_failures.con...          990     1647815400     1647817140  8.466500e+02  0.000000e+00  2.047273e+02             50                1.647815e+09         1.454500e+03         0.000000e+00              high
16  container.node-5.container_memory_failures.hie...          990     1647815400     1647817140  8.466500e+02  0.000000e+00  2.047273e+02             50                1.647815e+09         1.454500e+03         0.000000e+00              high
17  container.node-5.container_cpu_cfs_throttled_s...          990     1647815400     1647817140  9.408858e-03  0.000000e+00  8.508838e-02             50                1.647815e+09         3.402247e+01         0.000000e+00              high
18        container.node-5.container_cpu_user_seconds          990     1647815400     1647817140  1.700000e-01  0.000000e+00  1.048333e-01             49                1.647815e+09         2.447500e+01         0.000000e+00              high
19      container.node-5.container_cpu_system_seconds          990     1647815400     1647817140  8.000000e-02  0.000000e+00  1.706061e-02             47                1.647815e+09         1.250000e-01         0.000000e+00              high, True, 0     container.node-5.container_network_receive_MB....
1     container.node-5.container_network_receive_err...
2     container.node-5.container_network_receive_pac...
3     container.node-5.container_network_receive_pac...
4     container.node-5.container_network_transmit_MB...
                            ...                        
19                             system.net.udp.in_errors
20                         system.net.udp.out_datagrams
21                        system.net.udp.rcv_buf_errors
22                        system.net.udp.snd_buf_errors
23                               system.tcp.retrans_pct
Name: matching_kpis, Length: 24, dtype: object)```
```